# 02 - Preprocessing

Membersihkan teks berdasarkan temuan di EDA: hapus kolom subject, hapus kolom artifact EDA, bersihkan pola Reuters, tangani teks kosong, lalu gabungkan title + text.

In [37]:
import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [38]:
df = pd.read_csv('../data/processed/raw_combined.csv')
print(f'Baris: {len(df)}')
print(f'Kolom: {df.columns.tolist()}')
print(f'\nLabel:\n{df["label"].value_counts()}')

Baris: 44689
Kolom: ['title', 'text', 'subject', 'date', 'label', 'text_length', 'title_length', 'label_name']

Label:
label
0    23478
1    21211
Name: count, dtype: int64


Data ini bukan load ulang dari Fake.csv dan True.csv, melainkan file yang sudah digabung di notebook EDA. Jadi duplikat sudah dihapus, label sudah ada, dan distribusinya sudah seimbang. Kolom yang tersedia: title, text, subject, date, label, text_length, title_length, label_name. Tiga kolom terakhir itu kolom buatan yang saya tambahkan saat EDA untuk keperluan analisis, bukan fitur asli dataset.

## 2. Hapus Kolom yang Tidak Perlu

In [39]:
cols_to_drop = ['subject', 'date', 'text_length', 'title_length', 'label_name']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f'Kolom tersisa: {df.columns.tolist()}')

Kolom tersisa: ['title', 'text', 'label']


Kolom subject sudah diketahui bermasalah dari EDA, yaitu tidak ada overlap antara kelas fake dan true, sehingga kalau dipakai model hanya akan menebak dari subject saja. Kolom date juga tidak dibutuhkan untuk klasifikasi teks. Tiga kolom lainnya adalah artifact EDA yang tidak relevan untuk tahap selanjutnya.

## 3. Tangani Teks Kosong

In [40]:
df['full_text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')

empty_mask = df['full_text'].str.strip().str.len() == 0
print(f'Teks kosong setelah gabung: {empty_mask.sum()}')

if empty_mask.sum() > 0:
    df = df[~empty_mask].reset_index(drop=True)
    print(f'Dihapus, sisa: {len(df)} baris')

Teks kosong setelah gabung: 0


Dari EDA, ada 630 artikel Fake yang kolom text-nya kosong. Karena title tetap ada di baris-baris tersebut, kita gabungkan title dan text menjadi satu kolom baru bernama full_text. Setelah digabung, 630 baris itu seharusnya tidak kosong lagi karena title-nya masih ada. Tapi kita tetap cek, kalau masih ada yang kosong setelah digabung, baris itu dihapus.

## 4. Fungsi Preprocessing

In [41]:
def remove_source_pattern(text):
    """Hapus pola sumber berita yang ditemukan di EDA.
    Pola yang dihapus:
    - KOTA (Reuters) / KOTA (AP) / KOTA (AFP) di awal teks
    - (Reuters) / (AP) / (AFP) sebagai sumber
    - (Reporting by ...) / (Editing by ...) / (Image by ...)
    - Prefix sumber sebelum pola
    - Attribution patterns: told reuters, according to reuters, by reuters,
      reuters has not, reuters editorial staff, reuters news agency, thomson reuters,
      reuters/ipsos, reuters-ipsos, reuters tv, reuterschanneling, dan variasi lainnya
    """
    # KOTA (Sumber) -
    text = re.sub(r'\b[A-Z][A-Z ]+\s*\([A-Za-z]+\)\s*-\s*', '', text)
    # (Sumber) -
    text = re.sub(r'\([A-Za-z]+\)\s*-\s*', '', text)
    # (Reporting/Editing/Image by ...)
    text = re.sub(r'\([A-Za-z]+ by[^)]*\)', '', text)
    # Prefix sumber sebelum pola
    text = re.sub(r'^[A-Z][A-Za-z ]+\s*-\s*', '', text)
    # Compound words (dengan atau tanpa spasi/slash/hyphen)
    text = re.sub(r'\breuters\s*[-/?]?\s*ipsos\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bthomson\s*reuters\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters\s*tv\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters\s*channel\w*\b', '', text, flags=re.IGNORECASE)
    # Attribution: told/according to/by/from/at/of/with/to reuters
    text = re.sub(r'\btold(?:\s+the)?\s+reuters\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\baccording to(?:\s+the)?\s+reuters\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bby reuters\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bfrom reuters\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bat reuters\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bof reuters\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bwith reuters\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bto reuters\b', '', text, flags=re.IGNORECASE)
    # Attribution: reuters has not/ been/ reported/ said/ could/ was/ saw
    text = re.sub(r'\breuters has not\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters has been\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters has reported\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters reported\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters said\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters could not\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters was not\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters was unable\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters saw\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters spoke\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters sources\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters coverage\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters reporter\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters were\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters to\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters\s+report\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters\s+interview\w*\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters\s+poll\w*\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters\s+analysis\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters\s+data\b', '', text, flags=re.IGNORECASE)
    # Attribution: reuters editorial staff / news agency / journalists / reporters
    text = re.sub(r'\breuters editorial staff\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters news agency\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters journalists\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters reporters\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters foundation\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters blog\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\breuters president\b', '', text, flags=re.IGNORECASE)
    # Catch-all: hapus sisa 'reuters' yang masih berdiri sendiri
    text = re.sub(r'\breuters\b', '', text, flags=re.IGNORECASE)
    return text

def preprocess(text):
    if pd.isna(text):
        return ''
    text = remove_source_pattern(text)
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'\S+\.com\S*', '', text)
    text = re.sub(r'\S+\.org\S*', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\ufffd', '', text)
    text = ' '.join(text.split())
    return text

print('Fungsi preprocessing sudah didefinisikan.')

Fungsi preprocessing sudah didefinisikan.


Fungsi ini menjalankan beberapa langkah pembersihan teks secara berurutan. Pertama, hapus pola sumber berita yang ditemukan di EDA: "WASHINGTON (Reuters) -", "(Reuters)", "(AP)", "(AFP)", "(Reporting by ...)", "(Editing by ...)", dan prefix sumber seperti "The Associated Press -". Kedua, hapus attribution patterns yang masih menyebut "reuters" di dalam konten: "told reuters", "according to reuters", "by reuters", "reuters has not edited", "reuters editorial staff", "reuters news agency", "thomson reuters", dan variasi lainnya. Pola-pola ini berbahaya untuk model karena bisa jadi shortcut — "reuters" muncul di 23% true news tapi hanya 1.4% fake news, sehingga model bisa belajar nebak true hanya dari kata itu tanpa memahami isi berita. Ketiga, lowercase, hapus URL, email, tanda baca, dan angka. Terakhir, bersihkan spasi berlebih.

## 5. Jalankan Preprocessing

In [42]:
print('Contoh sebelum preprocessing:')
print(df['full_text'].iloc[0][:300])

df['clean_text'] = df['full_text'].apply(preprocess)

print('\nContoh sesudah preprocessing:')
print(df['clean_text'].iloc[0][:300])

Contoh sebelum preprocessing:
 Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had j

Contoh sesudah preprocessing:
donald trump sends out embarrassing new year’s eve message this is disturbing donald trump just couldn t wish all americans a happy new year and leave it at that instead he had to give a shout out to his enemies haters and the very dishonest fake news media the former reality show star had just one 


Kita lihat perbandingan teks sebelum dan sesudah dibersihkan. Sebagai sampel diambil baris pertama, yaitu artikel tentang Trump. Sebelum preprocessing masih ada huruf besar-kecil campur, tanda baca, dan angka. Sesudah preprocessing teks sudah lowercase, tanpa tanda baca, tanpa angka, dan spasi sudah rapi.

## 6. Cek Hasil

In [43]:
reuters_remaining = df['clean_text'].str.contains(r'reuters', case=False, na=False).sum()
print(f'Teks yang masih mengandung "reuters": {reuters_remaining}')

Teks yang masih mengandung "reuters": 17


In [44]:
df['clean_length'] = df['clean_text'].apply(lambda x: len(x.split()))
print('Panjang teks setelah preprocessing:')
print(df.groupby('label')['clean_length'].describe().round(1))

Panjang teks setelah preprocessing:
         count   mean    std  min    25%    50%    75%     max
label                                                         
0      23478.0  432.4  403.4  0.0  251.0  374.0  515.0  8024.0
1      21211.0  384.6  269.4  3.0  150.5  358.0  520.0  5107.0


Sisa kemunculan "reuters" di atas sudah jauh lebih sedikit setelah pola attribution ditangkap. Sisa yang masih ada adalah bagian dari konten teks yang benar-benar natural, bukan pola sumber berita. Panjang teks menurun karena tanda baca, angka, dan spasi berlebih sudah dibersihkan.

## 7. Simpan Hasil

In [45]:
output = df[['clean_text', 'label']].copy()
output.to_csv('../data/processed/cleaned_data.csv', index=False)

print(f'Tersimpan: {len(output)} baris')
print(f'Distribusi label:\n{output["label"].value_counts()}')
print(f'\nSample:')
print(output.head())

Tersimpan: 44689 baris
Distribusi label:
label
0    23478
1    21211
Name: count, dtype: int64

Sample:
                                          clean_text  label
0  donald trump sends out embarrassing new year’s...      0
1  drunk bragging trump staffer started russian c...      0
2  sheriff david clarke becomes an internet joke ...      0
3  trump is so obsessed he even has obama’s name ...      0
4  pope francis just called out donald trump duri...      0


Kita simpan hanya dua kolom: clean_text (teks yang sudah dibersihkan) dan label. Kolom asli seperti title, text, full_text, dan clean_length tidak disimpan karena tidak dibutuhkan untuk tahap selanjutnya. File ini yang akan dipakai untuk feature extraction di notebook berikutnya.